# 01 — Rolling Window Setup

Partitions the processed dataset into chronological time steps and enumerates all valid (Model A, Model B) window pairs for the rolling-retraining experiment.

**Input:** `data/processed/` (from notebook 00)  
**Output:** `data/windows/window_config.json`

---

**Framework recap (§3.1):**
- Data partitioned into time steps D_{t1}, …, D_{tK} (one per calendar week).
- Training window: W_k = D_{t_{k-L+1}} ∪ … ∪ D_{t_k}  (fixed length L).
- Model A trained on W_k, Model B on W_{k+s}.
- Common evaluation slice: E_{A,B} = D_{t_{k+s+1}} ∪ … ∪ D_{t_{k+s+h}}.

**Parameters used:**
| Parameter | Value | Meaning |
|-----------|-------|--------|
| L | 6 | training window length (weeks) |
| s | 1 | step between windows |
| h | 2 | evaluation horizon (weeks) |
| R | 3 | replicas per window |
| τ | 0.5 | decision threshold |

In [ ]:
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

WORKSPACE = Path('/content/drive/MyDrive/Thesis/Shoppers_workspace')
PROC_DIR  = WORKSPACE / 'data' / 'processed'
WIN_DIR   = WORKSPACE / 'data' / 'windows'
WIN_DIR.mkdir(parents=True, exist_ok=True)

# ── Experiment parameters ────────────────────────────────────────────────
L = 6    # training window length (time steps)
S = 1    # step between Model A and Model B windows
H = 2    # evaluation horizon (time steps)
R = 3    # replicas per window
TAU = 0.5  # decision threshold

print(f'Parameters: L={L}, S={S}, H={H}, R={R}, τ={TAU}')

## 1. Load processed data

In [ ]:
X    = pd.read_parquet(PROC_DIR / 'X.parquet')
Y    = np.load(PROC_DIR / 'Y.npy')
meta = pd.read_parquet(PROC_DIR / 'meta.parquet')

# Add a global row index for later reference
meta = meta.reset_index(drop=True)
meta['row_idx'] = meta.index

print(f'X: {X.shape}, Y: {Y.shape}')
print(f'Date range: {meta["offerdate"].min()} → {meta["offerdate"].max()}')

## 2. Define time steps (calendar weeks)

In [ ]:
# Assign each row to an ISO calendar week  
meta['week'] = meta['offerdate'].dt.to_period('W').dt.start_time

# Count offers per week
week_counts = meta.groupby('week').size().sort_index()
print('Offers per calendar week:')
print(week_counts.to_string())

fig, ax = plt.subplots(figsize=(12, 3))
week_counts.plot(ax=ax, kind='bar', color='steelblue', width=0.8)
ax.set_title('Offers per calendar week')
ax.set_xlabel('Week start')
ax.set_ylabel('Count')
plt.xticks(rotation=45, ha='right', fontsize=8)
plt.tight_layout()
plt.savefig(WIN_DIR / 'offers_per_week.png', dpi=120)
plt.show()

In [ ]:
# Ordered list of distinct week labels (time steps t_1 ... t_K)
time_steps = sorted(week_counts.index.tolist())
K = len(time_steps)
print(f'K = {K} time steps')
for i, ts in enumerate(time_steps):
    n = week_counts[ts]
    print(f'  t_{i+1}: {ts.strftime("%Y-%m-%d")}  ({n:,} offers)')

In [ ]:
# If certain weeks are very sparse (< 50 offers), merge them with adjacent week
# or switch to biweekly granularity.
MIN_PER_STEP = 50
sparse_weeks = [ts for ts, n in week_counts.items() if n < MIN_PER_STEP]
if sparse_weeks:
    print(f'WARNING: {len(sparse_weeks)} week(s) have < {MIN_PER_STEP} offers: {sparse_weeks}')
    print('Consider switching to biweekly granularity in the parameter cell above.')
else:
    print('All weeks have sufficient offers.')

# Map each row to its time-step index (0-based)
week_to_step = {w: i for i, w in enumerate(time_steps)}
meta['step'] = meta['week'].map(week_to_step)

## 3. Enumerate valid window pairs

A pair (A, B) is valid when:
- k ≥ L  (enough history for training window A)
- k + S + H ≤ K − 1  (evaluation slice fits within the data)

where k is the last time step of window A (0-based).

In [ ]:
def get_indices(step_set: set) -> list:
    """Row indices for the given set of time-step indices."""
    return meta[meta['step'].isin(step_set)]['row_idx'].tolist()


pairs = []

for k in range(L - 1, K):           # k = last step of window A (0-based)
    k_b = k + S                      # last step of window B
    eval_start = k_b + 1             # first step of evaluation slice
    eval_end   = k_b + H             # last step of evaluation slice (inclusive)

    if eval_end >= K:
        break                        # not enough future data

    # Time-step sets
    steps_A    = set(range(k - L + 1, k + 1))        # W_k
    steps_B    = set(range(k_b - L + 1, k_b + 1))    # W_{k+s}
    steps_eval = set(range(eval_start, eval_end + 1)) # E_{A,B}

    # Row index lists
    idx_A    = get_indices(steps_A)
    idx_B    = get_indices(steps_B)
    idx_eval = get_indices(steps_eval)

    if len(idx_eval) == 0:
        print(f'Skipping pair k={k}: empty evaluation slice')
        continue

    pairs.append({
        'pair_id':    len(pairs),
        'k':          k,
        'k_b':        k_b,
        'step_label_A':   time_steps[k].strftime('%Y-%m-%d'),
        'step_label_B':   time_steps[k_b].strftime('%Y-%m-%d'),
        'eval_start_label': time_steps[eval_start].strftime('%Y-%m-%d'),
        'eval_end_label':   time_steps[eval_end].strftime('%Y-%m-%d'),
        'steps_A':    sorted(steps_A),
        'steps_B':    sorted(steps_B),
        'steps_eval': sorted(steps_eval),
        'idx_A':      idx_A,
        'idx_B':      idx_B,
        'idx_eval':   idx_eval,
        'n_train_A':  len(idx_A),
        'n_train_B':  len(idx_B),
        'n_eval':     len(idx_eval),
    })

print(f'\nTotal valid window pairs: {len(pairs)}')

In [ ]:
# Summary table
summary = pd.DataFrame([{
    'pair_id':  p['pair_id'],
    'A_window': f"{p['step_label_A']}",
    'B_window': f"{p['step_label_B']}",
    'eval_period': f"{p['eval_start_label']} → {p['eval_end_label']}",
    'n_train_A': p['n_train_A'],
    'n_train_B': p['n_train_B'],
    'n_eval':    p['n_eval'],
} for p in pairs])

print(summary.to_string(index=False))

# Check for empty evaluation slices — these must not exist
empty_eval = summary[summary['n_eval'] == 0]
if not empty_eval.empty:
    print(f'\nWARNING: {len(empty_eval)} pairs with empty evaluation slice!')
else:
    print('\nAll evaluation slices are non-empty.')

## 4. Verify non-overlap of evaluation slices

For a fair experiment the evaluation slice must chronologically follow both training windows.

In [ ]:
for p in pairs:
    # Evaluation steps must not overlap with training steps
    assert not set(p['steps_eval']) & set(p['steps_A']), f"Pair {p['pair_id']}: eval overlaps A!"
    assert not set(p['steps_eval']) & set(p['steps_B']), f"Pair {p['pair_id']}: eval overlaps B!"
    # All eval steps must come after all training steps
    assert min(p['steps_eval']) > max(p['steps_B']), f"Pair {p['pair_id']}: eval not after B!"

print('Non-overlap checks passed for all pairs.')

## 5. Save window configuration

In [ ]:
config = {
    'parameters': {
        'L': L, 'S': S, 'H': H, 'R': R, 'TAU': TAU,
        'K': K,
        'time_steps': [ts.strftime('%Y-%m-%d') for ts in time_steps],
    },
    'pairs': pairs
}

out_path = WIN_DIR / 'window_config.json'
with open(out_path, 'w') as f:
    json.dump(config, f, indent=2)

size_kb = out_path.stat().st_size / 1024
print(f'Saved {out_path.name} ({size_kb:.0f} KB)')
print(f'{len(pairs)} window pairs, {K} time steps')

In [ ]:
# Visual: training and eval set sizes over time
fig, axes = plt.subplots(1, 2, figsize=(12, 3))

pair_ids = [p['pair_id'] for p in pairs]
axes[0].plot(pair_ids, [p['n_train_A'] for p in pairs], 'o-', label='Train A')
axes[0].plot(pair_ids, [p['n_train_B'] for p in pairs], 's--', label='Train B')
axes[0].set_title('Training set sizes')
axes[0].set_xlabel('Window pair')
axes[0].set_ylabel('# instances')
axes[0].legend()

axes[1].bar(pair_ids, [p['n_eval'] for p in pairs], color='darkorange')
axes[1].set_title('Evaluation slice sizes')
axes[1].set_xlabel('Window pair')
axes[1].set_ylabel('# instances')

plt.tight_layout()
plt.savefig(WIN_DIR / 'window_sizes.png', dpi=120)
plt.show()
print('Done.')